# Adaptive AM-FM Decomposition of Speech for Parkinson's Disease Classification
## XGBoost classification on the NeuroVoz vowels

Classifies speakers with Parkinson's disease (PD) versus healthy controls (HC) from sustained vowels of the NeuroVoz corpus, using eaQHM harmonic AM/FM variation features ($H_1$–$H_5$) together with the normalised first differences of $f_0$ and $A_1$ and spectral / Teager-energy descriptors (18 features). `scale_pos_weight` is set to the speaker-level HC/PD ratio.

The evaluation protocol matches the other NeuroVoz notebooks: speaker-independent folds stratified by label and gender (5 repeats × 10 outer folds), loaded from `master_cv_folds_neurovoz_vowels.csv` or generated and saved if that file does not exist, with a 5-fold inner grid search over the number of trees, learning rate, maximum depth, row subsampling and column subsampling (ROC AUC). Results are reported at sample and speaker level.

### Imports

In [1]:
import pandas as pd
import numpy as np
from scipy.io import loadmat
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_score, recall_score,
                             log_loss, brier_score_loss)
from typing import Tuple
import os

### 0. Data preparation — load features and parse file names

In [2]:
# ==========================================
# 0. DATA PREP
# ==========================================
task = "neurovoz_vowels"

raw = loadmat('../features/neurovoz_16k_5ms_chopped_650.mat')
data_raw = raw['results'].squeeze()
data = pd.DataFrame(data_raw)
data.columns = ['centroid_mean', 'centroid_std', 'spectral_flux_mean', 'spectral_flux_max',
                'teo_mean', 'teo_std', 'am_fm_corr', 'ampl_var', 'freq_var', 'f0_var',
                'SRER', 'f0_norm_diff', 'jitter_T', 'A1_norm_diff', 'spectral_slope',
                'f0_entropy', 'name']

data['name'] = data['name'].apply(lambda x: x[0])

for col in ['centroid_mean', 'centroid_std', 'spectral_flux_mean', 'spectral_flux_max',
            'teo_mean', 'teo_std', 'am_fm_corr', 'f0_var', 'SRER',
            'f0_norm_diff', 'jitter_T', 'A1_norm_diff', 'spectral_slope', 'f0_entropy']:
    data[col] = data[col].apply(lambda x: x[0][0])

filename_parts  = data['name'].str.split('_')
data['label']   = filename_parts.str[0].map({'PD': 1, 'HC': 0})
data['speaker']  = filename_parts.str[2]
data['vowel']    = filename_parts.str[1].str[0].str.upper()

### 0a. Gender metadata

In [ ]:
# Gender metadata 
metadata_hc = pd.read_csv("../gender_metadata/metadata_hc_neurovoz.csv")
metadata_pd = pd.read_csv("../gender_metadata/metadata_pd_neurovoz.csv")

gender_data = pd.concat([metadata_hc[['ID', 'Sex']], metadata_pd[['ID', 'Sex']]])
gender_data = gender_data.dropna(subset=['Sex']).drop_duplicates()
gender_data = gender_data.rename(columns={'ID': 'speaker', 'Sex': 'gender'})
gender_data['speaker'] = gender_data['speaker'].astype(int).astype(str).str.zfill(4)
gender_data['gender']  = gender_data['gender'].astype(int)

data['speaker'] = data['speaker'].astype(str)
data = data.merge(gender_data, how='left', on='speaker')

### 0b. Manual gender fixes

In [ ]:
# Manual gender fixes 
manual_genders = {'0068': 0, '0069': 0, '0084': 0, '0121': 1, '0059': 0}
data['gender'] = data['gender'].fillna(data['speaker'].map(manual_genders))
data['gender'] = data['gender'].astype(int)

### 0c. Remove bad / corrupted recordings

In [ ]:
# Remove bad/corrupted files 
bad_files = [
    'HC_A1_0084', 'HC_A1_0087', 'HC_A2_0068', 'HC_A2_0071',
    'HC_A2_0084', 'HC_A2_0087', 'HC_E2_0071', 'HC_E2_0087',
    'HC_E3_0034', 'HC_I1_0071', 'HC_I1_0084', 'HC_I2_0049',
    'HC_I2_0063', 'HC_I2_0083', 'HC_I2_0084', 'HC_I2_0087',
    'HC_O1_0071', 'HC_O1_0083', 'HC_O1_0084', 'HC_O2_0071',
    'HC_O2_0087', 'HC_U1_0053', 'HC_U1_0071', 'HC_U1_0083',
    'HC_U1_0084', 'HC_U1_0087', 'HC_U2_0144', 'PD_A1_0078',
    'PD_E2_0014', 'PD_E2_0018', 'PD_E2_0022', 'PD_E2_0023',
    'PD_E2_0024', 'PD_E2_0077', 'PD_E2_0078', 'PD_E2_0079',
    'PD_E3_0004', 'PD_E3_0007', 'PD_E3_0008', 'PD_E3_0011',
    'PD_E3_0012', 'PD_O1_0007', 'PD_O1_0009', 'PD_O1_0010',
    'PD_O1_0014', 'PD_O1_0015', 'PD_O1_0070',
]
initial_count = len(data)
data = data[~data['name'].isin(bad_files)].reset_index(drop=True)
print(f"Removed {initial_count - len(data)} bad files. Remaining: {len(data)}")

Removed 47 bad files. Remaining: 1019


### 0d. Expand harmonic features and select features

In [ ]:
# Expand harmonic features
ampl_expanded = data['ampl_var'].apply(lambda x: x.flatten())
freq_expanded = data['freq_var'].apply(lambda x: x.flatten())
for i in range(5):
    data[f'ampl_var_H{i+1}'] = ampl_expanded.apply(lambda v: v[i])
    data[f'freq_var_H{i+1}'] = freq_expanded.apply(lambda v: v[i])

data = data.drop(columns=['jitter_T', 'ampl_var', 'freq_var', 'am_fm_corr',
                           'f0_entropy', 'f0_var', 'SRER', 'spectral_slope'])

### 0e. Sanity checks

In [7]:
# ── Sanity checks ──────────────────────────────────────────────────────────────
group_prefix   = data['name'].str.split('_').str[0]
expected_label = group_prefix.map({'PD': 1, 'HC': 0})
assert (data['label'] == expected_label).all(), "Label mismatch detected!"
assert data.groupby('speaker')['label'].nunique().max() == 1, "Speaker has inconsistent labels!"

print(f"\nData shape     : {data.shape}")
print(f"Unique speakers: {data['speaker'].nunique()}")
print("Class distribution (sample-level):"); print(data['label'].value_counts())
print("Class distribution (speaker-level):"); print(data[['speaker','label']].drop_duplicates()['label'].value_counts())
print("Gender distribution (0=F, 1=M):"); print(data[['speaker','gender']].drop_duplicates()['gender'].value_counts())
print(f"Total NaNs     : {data.isna().sum().sum()}")


Data shape     : (1019, 23)
Unique speakers: 111
Class distribution (sample-level):
label
1    545
0    474
Name: count, dtype: int64
Class distribution (speaker-level):
label
0    58
1    53
Name: count, dtype: int64
Gender distribution (0=F, 1=M):
gender
1    61
0    50
Name: count, dtype: int64
Total NaNs     : 0


### 0f. Class-balance weight for XGBoost

In [8]:
# ── Class balance weight for XGBoost ──────────────────────────────────────────
n_neg = (data[['speaker','label']].drop_duplicates()['label'] == 0).sum()
n_pos = (data[['speaker','label']].drop_duplicates()['label'] == 1).sum()
scale_pos_weight = n_neg / n_pos
print(f"\nSpeaker-level class ratio (HC/PD): {scale_pos_weight:.4f}  → used as scale_pos_weight")


Speaker-level class ratio (HC/PD): 1.0943  → used as scale_pos_weight


### 1. Load or create cross-validation folds

In [ ]:
# ==========================================
# 1. LOAD OR CREATE FOLDS
# ==========================================
fold_file = f"../folds/master_cv_folds_{task}.csv"

N_REPEATS      = 5
N_OUTER_SPLITS = 10
N_INNER_SPLITS = 5
base_random_state = 42

data['stratify_key'] = data['label'].astype(str) + "_" + data['gender'].astype(str)

if os.path.exists(fold_file):
    # Load existing folds 
    print(f"\nLoading predefined folds from {fold_file}...")
    fold_map = pd.read_csv(fold_file)
    fold_map['speaker'] = fold_map['speaker'].astype(str).str.strip().str.zfill(4)

    repeat_fold_cols = [c for c in fold_map.columns if c.startswith("Repeat_") and c.endswith("_Fold")]
    speaker_folds    = fold_map[['speaker'] + repeat_fold_cols].drop_duplicates(subset='speaker')

    data['speaker'] = data['speaker'].astype(str).str.strip().str.zfill(4)
    data = data.merge(speaker_folds, on='speaker', how='inner').reset_index(drop=True)

    if data.empty:
        raise ValueError("CRITICAL: DataFrame empty after fold merge — check speaker ID format")

    for col in repeat_fold_cols:
        assert data.groupby('speaker')[col].nunique().max() == 1, \
            f"Speaker has inconsistent fold assignments in {col}"
    print("✅ All speakers have consistent fold assignments")

    original_speakers = set(fold_map['speaker'].str.strip().str.zfill(4))
    current_speakers  = set(data['speaker'].str.strip().str.zfill(4))
    missing = current_speakers - original_speakers
    if missing:
        print(f"⚠️  {len(missing)} speakers not in fold file: {missing}")

    N_REPEATS      = len(repeat_fold_cols)
    N_OUTER_SPLITS = data[repeat_fold_cols[0]].nunique()

else:
    # Generate and save new folds 
    print(f"\nNo fold file found. Generating new folds and saving to {fold_file}...")

    data['speaker'] = data['speaker'].astype(str).str.strip().str.zfill(4)
    groups_for_cv   = data['speaker'].values
    y_stratify_cv   = data['stratify_key']
    X_tmp = data.drop(columns=['label', 'gender', 'stratify_key', 'speaker',
                                'name', 'vowel'], errors='ignore')

    fold_map = pd.DataFrame({
        'sample_id': np.arange(len(data)),
        'speaker':   groups_for_cv,
        'label':     data['label'].values,
    })
    repeat_fold_cols = [f'Repeat_{r+1}_Fold' for r in range(N_REPEATS)]
    for col in repeat_fold_cols:
        fold_map[col] = -1

    for repeat in range(N_REPEATS):
        current_seed = base_random_state + repeat
        outer_cv = StratifiedGroupKFold(
            n_splits=N_OUTER_SPLITS, shuffle=True, random_state=current_seed
        )
        for fold, (_, test_idx) in enumerate(outer_cv.split(X_tmp, y_stratify_cv, groups_for_cv)):
            fold_map.iloc[test_idx, fold_map.columns.get_loc(f'Repeat_{repeat+1}_Fold')] = fold

    assert (fold_map[repeat_fold_cols] == -1).sum().sum() == 0, \
        "Some samples were never assigned to a fold!"

    fold_map.to_csv(fold_file, index=False)
    print(f"✅ Saved {fold_file} ({len(fold_map)} rows, {fold_map['speaker'].nunique()} speakers)")

    speaker_folds = fold_map[['speaker'] + repeat_fold_cols].drop_duplicates(subset='speaker')
    data = data.merge(speaker_folds, on='speaker', how='inner').reset_index(drop=True)


Loading predefined folds from ../folds/master_cv_folds_neurovoz_vowels.csv...
✅ All speakers have consistent fold assignments


### 2. Features and groups

In [ ]:
# ==========================================
# 2. FINALIZE FEATURES AND GROUPS
# ==========================================
groups = data['speaker'].values

cols_to_drop = ['label', 'gender', 'stratify_key', 'speaker', 'name',
                'vowel', 'sample_id'] + repeat_fold_cols
X = data.drop(columns=cols_to_drop, errors='ignore')
y = data['label']
y_stratify = data['stratify_key']

feature_cols = list(X.columns)

print(f"\nFeature matrix : {X.shape}")
print(f"Features       : {feature_cols}")
print(f"Unique speakers: {len(np.unique(groups))}")
print(f"Repeats x Folds: {N_REPEATS} x {N_OUTER_SPLITS}")
print("Stratify key counts:"); print(y_stratify.value_counts().sort_index())

### 3. Helper functions

In [ ]:
# ==========================================
# 3. HELPERS
# ==========================================
def aggregate_mean_by_group(
    y: np.ndarray, p: np.ndarray, g: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    y, p, g = np.asarray(y).astype(int), np.asarray(p).astype(float), np.asarray(g)
    uniq = np.unique(g)
    y_g  = np.zeros(len(uniq), dtype=int)
    p_g  = np.zeros(len(uniq), dtype=float)
    for i, gg in enumerate(uniq):
        idx    = np.where(g == gg)[0]
        p_g[i] = float(np.mean(p[idx])) if len(idx) else float('nan')
        y_g[i] = int(np.mean(y[idx]) >= 0.5) if len(idx) else 0
    return y_g, p_g, uniq

### 4. Result storage

In [ ]:
# ==========================================
# 4. RESULT STORAGE
# ==========================================
metrics_keys = ["accuracy", "f1", "auc", "precision", "recall", "log_loss", "brier"]
results_sample  = {k: [] for k in metrics_keys}
results_speaker = {k: [] for k in metrics_keys}
conf_matrices_sample  = []
conf_matrices_speaker = []
all_perm_importances  = []

# Per-fold (fold-to-fold) records, one row per (repeat, outer fold)
fold_records = []

### 5. Repeated nested cross-validation

In [ ]:
# ==========================================
# 5. MAIN LOOP
# ==========================================
print(f"\nStarting XGBoost Repeated Nested CV ({N_REPEATS} Repeats × {N_OUTER_SPLITS} Folds)…")

for repeat_idx, repeat_col in enumerate(repeat_fold_cols):
    repeat       = repeat_idx + 1
    current_seed = base_random_state + repeat_idx

    print(f"\n{'='*80}")
    print(f"REPEAT {repeat}/{N_REPEATS}  (Seed: {current_seed} | Column: {repeat_col})")
    print("="*80)

    for fold_id in range(N_OUTER_SPLITS):

        test_mask  = (data[repeat_col] == fold_id).values
        train_mask = ~test_mask

        X_train, X_test = X.iloc[train_mask], X.iloc[test_mask]
        y_train, y_test = y.iloc[train_mask], y.iloc[test_mask]
        g_train_np      = groups[train_mask]
        g_test_np       = groups[test_mask]
        y_strat_train   = y_stratify.iloc[train_mask]

        # LEAKAGE CHECK 1: Outer speaker overlap 
        outer_overlap = set(g_train_np).intersection(set(g_test_np))
        assert len(outer_overlap) == 0, \
            f"❌ OUTER LEAKAGE R{repeat} Fold {fold_id}: {outer_overlap}"

        # LEAKAGE CHECK 2: Speaker counts 
        n_spk_train = len(set(g_train_np))
        n_spk_test  = len(set(g_test_np))
        assert n_spk_train + n_spk_test == len(np.unique(groups)), \
            f"❌ Speaker count mismatch: {n_spk_train}+{n_spk_test} != {len(np.unique(groups))}"

        # LEAKAGE CHECK 3: Sample index overlap 
        train_indices = np.where(train_mask)[0]
        test_indices  = np.where(test_mask)[0]
        assert len(set(train_indices) & set(test_indices)) == 0, \
            f"❌ SAMPLE OVERLAP R{repeat} Fold {fold_id}"

        # LEAKAGE CHECK 4: Forbidden columns in X
        forbidden   = ["label", "speaker", "speaker_id", "gender", "stratify_key", "name"]
        leaked_cols = [c for c in forbidden if c in X_train.columns]
        assert len(leaked_cols) == 0, \
            f"❌ Forbidden columns in features: {leaked_cols}"

        # LEAKAGE CHECK 5: Inner CV
        inner_cv_check = StratifiedGroupKFold(
            n_splits=N_INNER_SPLITS, shuffle=True,
            random_state=current_seed + fold_id,
        ).split(X_train, y_strat_train, g_train_np)

        for i, (tr_i, va_i) in enumerate(inner_cv_check):
            overlap_inner = set(g_train_np[tr_i]) & set(g_train_np[va_i])
            assert len(overlap_inner) == 0, \
                f"❌ INNER LEAKAGE R{repeat} Fold {fold_id} Inner {i}: {overlap_inner}"

        print(f"R{repeat} Fold {fold_id+1:2d}: ✅ All checks passed | "
              f"Train: {len(X_train)} samples ({n_spk_train} spk) | "
              f"Test: {len(X_test)} samples ({n_spk_test} spk)")

        # GridSearchCV 
        xgb = XGBClassifier(
            objective='binary:logistic',
            eval_metric='auc',
            scale_pos_weight=scale_pos_weight,
            random_state=current_seed,
            n_jobs=-1,
        )

        param_grid = {
            "n_estimators":    [100, 200, 300, 500],
            "learning_rate":   [0.01, 0.05, 0.1],
            "max_depth":       [3, 5, 7],
            "subsample":       [0.5, 0.8, 1.0],
            "colsample_bytree":[0.5, 0.8, 1.0],
        }

        inner_cv = StratifiedGroupKFold(
            n_splits=N_INNER_SPLITS, shuffle=True,
            random_state=current_seed + fold_id,
        ).split(X_train, y_strat_train, g_train_np)

        grid_search = GridSearchCV(
            estimator=xgb,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="roc_auc",
            n_jobs=-1,
            verbose=0,
        )
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        bp         = grid_search.best_params_

        # Sample-level predictions 
        y_pred_sample  = best_model.predict(X_test)
        y_proba_sample = best_model.predict_proba(X_test)[:, 1]

        results_sample["accuracy"].append(  accuracy_score(  y_test, y_pred_sample))
        results_sample["f1"].append(        f1_score(        y_test, y_pred_sample, zero_division=0))
        results_sample["auc"].append(       roc_auc_score(   y_test, y_proba_sample))
        results_sample["precision"].append( precision_score( y_test, y_pred_sample, zero_division=0))
        results_sample["recall"].append(    recall_score(    y_test, y_pred_sample, zero_division=0))
        results_sample["log_loss"].append(  log_loss(        y_test, y_proba_sample))
        results_sample["brier"].append(     brier_score_loss(y_test, y_proba_sample))
        conf_matrices_sample.append(confusion_matrix(y_test, y_pred_sample))

        # Speaker-level predictions
        y_test_spk, y_proba_spk, _ = aggregate_mean_by_group(
            y_test.values, y_proba_sample, g_test_np
        )
        y_pred_spk = (y_proba_spk >= 0.5).astype(int)

        results_speaker["accuracy"].append(  accuracy_score(  y_test_spk, y_pred_spk))
        results_speaker["f1"].append(        f1_score(        y_test_spk, y_pred_spk, zero_division=0))
        results_speaker["auc"].append(       roc_auc_score(   y_test_spk, y_proba_spk) if len(np.unique(y_test_spk)) > 1 else np.nan)
        results_speaker["precision"].append( precision_score( y_test_spk, y_pred_spk, zero_division=0))
        results_speaker["recall"].append(    recall_score(    y_test_spk, y_pred_spk, zero_division=0))
        results_speaker["log_loss"].append(  log_loss(        y_test_spk, y_proba_spk) if len(np.unique(y_test_spk)) > 1 else np.nan)
        results_speaker["brier"].append(     brier_score_loss(y_test_spk, y_proba_spk))
        conf_matrices_speaker.append(confusion_matrix(y_test_spk, y_pred_spk))

        fold_records.append({
            "repeat":                repeat,
            "fold":                  fold_id + 1,
            "seed":                  current_seed,
            "best_n_estimators":     bp['n_estimators'],
            "best_learning_rate":    bp['learning_rate'],
            "best_max_depth":        bp['max_depth'],
            "best_subsample":        bp['subsample'],
            "best_colsample_bytree": bp['colsample_bytree'],
            "n_train_samples":       len(X_train),
            "n_test_samples":        len(X_test),
            "n_train_speakers":      n_spk_train,
            "n_test_speakers":       n_spk_test,
            "sample_accuracy":       results_sample["accuracy"][-1],
            "sample_f1":             results_sample["f1"][-1],
            "sample_auc":            results_sample["auc"][-1],
            "sample_precision":      results_sample["precision"][-1],
            "sample_recall":         results_sample["recall"][-1],
            "sample_log_loss":       results_sample["log_loss"][-1],
            "sample_brier":          results_sample["brier"][-1],
            "speaker_accuracy":      results_speaker["accuracy"][-1],
            "speaker_f1":            results_speaker["f1"][-1],
            "speaker_auc":           results_speaker["auc"][-1],
            "speaker_precision":     results_speaker["precision"][-1],
            "speaker_recall":        results_speaker["recall"][-1],
            "speaker_log_loss":      results_speaker["log_loss"][-1],
            "speaker_brier":         results_speaker["brier"][-1],
        })

        print(f"           Best params: {bp}")
        print(f"           [Sample]  Acc: {results_sample['accuracy'][-1]:.4f} | "
              f"F1: {results_sample['f1'][-1]:.4f} | AUC: {results_sample['auc'][-1]:.4f}")
        print(f"           [Speaker] Acc: {results_speaker['accuracy'][-1]:.4f} | "
              f"F1: {results_speaker['f1'][-1]:.4f} | AUC: {results_speaker['auc'][-1]:.4f}")
        print("-" * 60)

### 6. Save per-fold results

In [ ]:
# ==========================================
# 6. SAVE FOLD-TO-FOLD RESULTS
# ==========================================
fold_df = pd.DataFrame(fold_records)

fold_csv = f"fold_results_xgb_{task}.csv"
fold_df.to_csv(fold_csv, index=False)
print(f"\nSaved per-fold results → {fold_csv}")

fold_txt = f"fold_results_xgb_{task}.txt"
with open(fold_txt, "w", encoding="utf-8") as f:
    f.write(f"XGBoost — ALL FEATURES ({len(feature_cols)} features) — {task}\n")
    f.write(f"Features: {feature_cols}\n")
    f.write(f"Repeats x Folds: {N_REPEATS} x {N_OUTER_SPLITS}\n")
    f.write("=" * 100 + "\n")
    f.write(fold_df.to_string(index=False))
    f.write("\n")
print(f"Saved per-fold results → {fold_txt}")

n_nan_auc = int(fold_df['speaker_auc'].isna().sum())
if n_nan_auc:
    print(f"⚠️  {n_nan_auc} fold(s) had a single-class speaker test set "
          f"→ speaker_auc / speaker_log_loss written as NaN")

### 7. Summary and confusion matrices

In [ ]:
# ==========================================
# 7. SUMMARY
# ==========================================
total_folds = N_REPEATS * N_OUTER_SPLITS
print(f"\n{'='*55}")
print(f"FINAL XGBoost RESULTS ({total_folds} Total Folds)")
print("="*55)
print(f"{'Metric':<12} | {'Sample-Level':<25} | {'Speaker-Level':<25}")
print("-" * 68)

for metric in metrics_keys:
    ms = np.nanmean(results_sample[metric]);  ss = np.nanstd(results_sample[metric])
    mk = np.nanmean(results_speaker[metric]); sk = np.nanstd(results_speaker[metric])
    print(f"{metric.capitalize():<12} | {ms:.4f} ± {ss:.4f}            | {mk:.4f} ± {sk:.4f}")

summary_rows = []
for level, r in (("sample", results_sample), ("speaker", results_speaker)):
    row = {"level": level}
    for metric in metrics_keys:
        row[f"{metric}_mean"] = np.nanmean(r[metric])
        row[f"{metric}_std"]  = np.nanstd(r[metric])
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_csv = f"results_xgb_{task}.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved aggregated summary → {summary_csv}")

print("\n--- Aggregated Confusion Matrix (Sample-Level) ---")
print(np.sum(conf_matrices_sample, axis=0))
print("\n--- Aggregated Confusion Matrix (Speaker-Level) ---")
print(np.sum(conf_matrices_speaker, axis=0))